# Elaboração do Dataset

Pipeline de ETL que transforma os dados brutos do FINBRA (CSV por ano)
em um único dataset consolidado, pronto pra análise.

**O que esse notebook faz:**
1. Descompacta os CSVs de `dados_compactos/` → `dados_extraidos/`
2. Lê cada CSV, limpa e enriquece com colunas derivadas
3. Consolida tudo em um único Parquet + tabela DuckDB

**Saída esperada:** `finbra_consolidado.parquet` + tabela `despesas_finbra` no DuckDB

## Pré-requisitos

Antes de rodar, garanta que o ambiente está configurado:
```bash
python -m venv venv
venv\Scripts\activate      # Windows
pip install -r requirements.txt
```

**Estrutura esperada do projeto:**
```
Desafio-Analista-de-Dados-Sefaz-Macei-
├── dados_compactos/          # ZIPs originais (2020-2025)
├── dados_extraidos/          # CSVs descompactados (gerado por este notebook)
├── data/processed/           # Parquet consolidado (gerado por este notebook)
├── finbra.duckdb             # Banco DuckDB (gerado por este notebook)
├── notebooks/
│   ├── 1-Preparar_Dataset.ipynb  # ← este notebook
│   └── 2-Analise.ipynb
├── src/
│   └── banco/
│       ├── conexao_duckdb.py
│       └── criar_tabela.py
└── requirements.txt
```

## 1. Configuração

Adiciona a raiz do projeto ao `sys.path` pra importar módulos de `src/`.

In [ ]:
import sys
import zipfile
import re
import time
from pathlib import Path

import pandas as pd
import duckdb

# Raiz do projeto (notebooks2/ → ../)
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))

from src.utils.constantes import CAMINHO_DUCKDB

pd.options.display.float_format = '{:,.2f}'.format

print(f'Raiz do projeto: {RAIZ}')
print(f'DuckDB destino:  {CAMINHO_DUCKDB}')

## 2. Descompactar ZIPs

Cada ano tem um ZIP em `dados_compactos/<ano>/`. Extraímos tudo pra `dados_extraidos/<ano>/`.

In [ ]:
from src.data_loader import extrair_zips

PASTA_COMPACTOS = RAIZ / 'dados_compactos'
PASTA_EXTRAIDOS = RAIZ / 'dados_extraidos'
PASTA_PROCESSED = RAIZ / 'data' / 'processed'

PASTA_EXTRAIDOS.mkdir(parents=True, exist_ok=True)
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

zips_encontrados = list(PASTA_COMPACTOS.rglob('*.zip'))
print(f'ZIPs encontrados: {len(zips_encontrados)}')
for caminho_zip in zips_encontrados:
    subpasta = caminho_zip.relative_to(PASTA_COMPACTOS).parent
    destino = PASTA_EXTRAIDOS / subpasta
    print(f'  Extraindo: {caminho_zip.name} -> {destino}')

extrair_zips(PASTA_COMPACTOS, PASTA_EXTRAIDOS)
print('Extração concluída.')

## 3. Classificação da Conta

O campo `Conta` do FINBRA segue padrões que indicam o nível de agregação:

| Padrão no texto | ContaTipo | Exemplo |
|---|---|---|
| Texto puro | `Total_Geral` | `Despesas Exceto Intraorçamentárias` |
| `DD - Texto` | `Função` | `10 - Saúde` |
| `DD.DDD - Texto` | `Subfunção` | `10.301 - Atenção Básica` |
| `FUDD - Texto` | `Subfuncao_agregada` | `FU10 - Demais Subfunções` |

**Por que importa?**
- `Função` já é o total das Subfunções → somar os dois gera duplicidade
- `Subfuncao_agregada` são subfunções abstraidas em um único valor
- `Total_Geral` é a soma de todas as funções

In [ ]:
from src.data_transformer import classificar_conta

# Teste rápido
testes = [
    'Despesas Exceto Intraorçamentárias',
    'Despesas Intraorçamentárias',
    '10 - Saúde',
    '10.301 - Atenção Básica',
    'FU10 - Demais Subfunções',
]
for t in testes:
    print(f'  {t:45s} -> {classificar_conta(t)}')

## 4. Leitura e Consolidação

Para cada CSV:
- Detecta o **ano** a partir da pasta (`dados_extraidos/2022/finbra.csv` → 2022)
- Lê com encoding `latin-1`, separador `;`, decimal `,`
- Pula as 3 primeiras linhas (metadados do Siconfi)
- Adiciona coluna `ContaTipo` classificando a conta

In [ ]:
from src.data_loader import carregar_e_consolidar_csvs
from src.data_transformer import descobrir_ano

arquivos_csv = sorted(PASTA_EXTRAIDOS.rglob('*.csv'))
print(f'CSVs encontrados: {len(arquivos_csv)}')
for caminho_csv in arquivos_csv:
    ano = descobrir_ano(caminho_csv)
    print(f'  Processando {ano}: {caminho_csv.name}')

df_consolidado = carregar_e_consolidar_csvs(PASTA_EXTRAIDOS)
print(f'\nConsolidado: {len(df_consolidado)} registros, {df_consolidado["Ano"].nunique()} anos')

## 5. Validação
Verificações rápidas pra garantir que os dados estão íntegros.

In [ ]:
print('=== CAPITAIS POR ANO ===')
capitais_por_ano = df_consolidado.groupby('Ano')['Instituição'].nunique()
for ano, qtd in capitais_por_ano.items():
    status = 'OK' if qtd == 26 else f'INCOMPLETO ({qtd}/26)'
    print(f'  {ano}: {qtd} capitais - {status}')

print('\n=== NULOS POR COLUNA ===')
nulos = df_consolidado.isnull().sum()
print(nulos[nulos > 0] if nulos.any() else '  Nenhum nulo encontrado.')

print('\n=== CONTA_TIPO DISTRIBUIÇÃO ===')
print(df_consolidado['ContaTipo'].value_counts().to_string())

print('\n=== COLUNAS DISPONÍVEIS ===')
print(list(df_consolidado.columns))

## 6. Salvar em Parquet

Formato colunar comprimido. Leitura rápida e occupying menos espaço.

In [ ]:
from src.data_loader import salvar_parquet

caminho_parquet = PASTA_PROCESSED / 'finbra_consolidado.parquet'
salvar_parquet(df_consolidado, caminho_parquet)
print(f'Parquet salvo: {caminho_parquet}')

## 7. Carregar no DuckDB

Cria a tabela física `despesas_finbra` no banco DuckDB.
Os notebooks de análise consultam essa tabela diretamente.

In [ ]:
from src.data_loader import carregar_no_duckdb

carregar_no_duckdb(caminho_parquet, CAMINHO_DUCKDB, 'despesas_finbra')

con = duckdb.connect(str(CAMINHO_DUCKDB))
total = con.execute('SELECT COUNT(*) FROM despesas_finbra').fetchone()[0]
print(f'Total de registros: {total}')

print('\nAmostra (3 linhas):')
print(con.execute('SELECT * FROM despesas_finbra LIMIT 3').df().to_string())

con.close()

## Conclusão

O dataset está pronto. Arquivos gerados:

| Arquivo | Caminho | Uso |
|---|---|---|
| Parquet | `data/processed/finbra_consolidado.parquet` | Leitura rápida |
| DuckDB | `finbra.duckdb` | Consultas SQL nos notebooks |

Agora é possível rodar `2-Analise.ipynb` diretamente.